# Hunyuan3D-2mv Geometry LoRA 학습4뷰 SD 캐릭터 이미지에서 형태가 무너지지 않는 3D 를 만들기 위한 Shape DiT LoRA 학습.**순서**: 설정 → 데이터 준비(GPU 4장 병렬) → 모델 로드 → LoRA 주입 → 학습 → 평가---### 이미 검증된 사실 (이 노트북을 쓰기 전에 실측으로 확인)| 항목 | 결과 ||---|---|| ShapeVAE 왕복 정확도 | chamfer 0.010 (정규화 -1~1). 수염·머리카락까지 보존 → **VAE 는 병목 아님** || DiT | 2.46B, frozen || LoRA rank 8 | 4.2M 학습 파라미터 (전체의 0.17%) || 속도 | 3090 1장 1,994 samples/h · 4장 7,993 samples/h (선형 확장) || 메모리 | gradient checkpointing 켜고 batch 8 에 12.6 GiB (끄면 batch 1 도 17.5 GiB) |### 반드시 지켜야 할 것 (전부 실제로 당하고 고친 것들)1. **LoRA 가중치는 fp32.** base 가 fp16 이라고 따라 쓰면 AdamW 의 eps(1e-8)가   fp16 최소 정규값(6.1e-5) 아래라 0 으로 언더플로 → **1스텝 만에 NaN**.2. **타임스텝은 fp32.** 모델 안에서 1000 을 또 곱하므로 fp16 으로 넘기면   sigma ≥ 0.07 에서 65504 를 넘어 NaN. (추론은 정수형이라 이 문제가 없다)3. **gradient checkpointing 필수.** 끄면 batch 1 도 24GB 를 넘긴다.4. **조건 임베딩은 `pipe.prepare_image` 경유.** `image_processor` 를 직접 부르면   배치 차원이 안 맞아 conditioner 에서 assert 로 죽는다.5. **평가 입력도 배경 제거 필수.** 흰 배경 RGB 를 그냥 넣으면 배경이 물체로 인식돼   정면이 평평한 판때기로 나온다.6. `Hunyuan3DDiT` 에는 `.dtype` 이 없다. `next(dit.parameters()).dtype` 을 쓸 것.### 학습 목적함수 (레포 스케줄러에서 확인)```x_t    = sigma * noise + (1 - sigma) * x0target = noise - x0t      = sigma * 1000        (fp32)loss   = MSE(DiT(x_t, t, cond), target)```

## 0. 환경 점검

In [ ]:
import os, sys, json, math, time, glob, random, subprocess, refrom pathlib import Pathimport numpy as np, torchREPO = Path('/data/sungju030307/img2glb/Hunyuan3D-2')HOME = Path('/home/sungju030307/img2glb')DATA = Path('/data/sungju030307/img2glb/train')os.environ.setdefault('HF_HUB_CACHE', '/data/sungju030307/hf-cache')sys.path.insert(0, str(REPO)); sys.path.insert(0, str(HOME))DATA.mkdir(parents=True, exist_ok=True)print('torch', torch.__version__, '| CUDA', torch.cuda.is_available(), '|', torch.cuda.device_count(), 'GPU')for i in range(torch.cuda.device_count()):    p = torch.cuda.get_device_properties(i)    print(f'  GPU {i}: {p.name} {p.total_memory/2**30:.0f} GiB')import torch_cluster; print('torch_cluster ok (ShapeVAE 인코더가 FPS 에 사용)')

## 1. 설정처음 돌릴 때는 `N_MODELS` 를 40 정도로 두고 끝까지 한 번 통과시킨 뒤 늘릴 것.

In [ ]:
CFG = dict(    # --- 데이터 ---    LIST       = '/data/sungju030307/objaverse_meta/hiprec_anime.json',   # 고정밀 애니메 1,229개    N_MODELS   = 400,        # 목록 앞에서 몇 개 시도할지 (통과율 약 60%)    NSHARDS    = 4,          # GPU 장수    RENDER_SIZE= 512,    CHAMFER_MAX= 0.025,      # VAE 재현 오차 상한 (품질 게이트)    MIN_FACES  = 20000,    # --- 뷰 규칙 ---    # Hunyuan3D-2mv view2idx = {front:0, left:1, back:2, right:3}    # 각도는 "물체를 Y축 +theta 회전 후 +Z 에서 본 것"    VIEW_AZIM  = {'front': 0, 'left': 90, 'back': 180, 'right': 270},    # --- LoRA ---    RANK       = 8,    ALPHA      = 16,    TARGETS    = 'attn',     # 'attn' | 'attn+mlp'    DROP_COND  = 0.1,        # CFG 유지를 위한 조건 드롭 확률    # --- 학습 ---    BATCH      = 8,          # 12.6 GiB. 부족하면 4    LR         = 1e-4,    EPOCHS     = 60,    CKPT_EVERY = 250,    RESUME     = True,       # ckpt/lora_last.pt 가 있으면 이어서    SEED       = 0,)CACHE = DATA/'cache'; CKPT = DATA/'ckpt'CACHE.mkdir(exist_ok=True); CKPT.mkdir(exist_ok=True)torch.manual_seed(CFG['SEED']); np.random.seed(CFG['SEED']); random.seed(CFG['SEED'])for k, v in CFG.items(): print(f'{k:12s} {v}')

## 2. 데이터 준비 (GPU 4장 병렬)`train/preprocess.py` 를 shard 로 나눠 GPU 장수만큼 동시에 돌린다.한 모델당 하는 일: 다운로드 → 정규화 → **VAE 재현 오차로 품질 게이트** → 4뷰 렌더 →GT latent 인코딩 → DINO 조건 임베딩 저장.품질 게이트가 핵심이다. 계획서에는 "watertight 여부로 필터링" 이라고 되어 있었는데실측해 보니 그게 기준이 아니었다. Objaverse 애니메 메시는 전부 non-watertight 인데도결과가 갈린다.| 모델 | faces | VAE 재현 chamfer | 결과 ||---|---|---|---|| Ghost Girl | 51,172 | 0.0094 | 좋음 || Human Charizard | 52,509 | 0.0102 | 좋음 || BB Yamato (우산·포신이 가늘다) | 200,000 | 0.0239 | 가는 부분 뭉개짐 || Laplus Lowpoly | 1,030 | 0.0371 | **사용 불가**, 완전히 노이즈 |즉 진짜 기준은 **ShapeVAE 가 그 메시를 표현할 수 있느냐**다. 이게 GT latent 품질의상한이므로 통과 못 하는 모델을 넣으면 LoRA 가 잡음을 배운다.한 모델당 25초쯤 걸리고 4장으로 나누면 1/4 이 된다. 중단 후 다시 실행해도이미 처리한 것은 건너뛴다.

In [ ]:
from IPython.display import clear_outputdef run_preprocess(n=None, nshards=None, timeout_min=600):    n = n or CFG['N_MODELS']; nshards = nshards or CFG['NSHARDS']    logs, procs = [], []    for sh in range(nshards):        lg = DATA/f'prep_{sh}.log'; logs.append(lg)        cmd = [sys.executable, str(HOME/'train/preprocess.py'),               '--list', CFG['LIST'], '--cache', str(CACHE), '--n', str(n),               '--nshards', str(nshards), '--shard', str(sh), '--gpu', str(sh),               '--size', str(CFG['RENDER_SIZE']), '--chamfer_max', str(CFG['CHAMFER_MAX']),               '--min_faces', str(CFG['MIN_FACES'])]        procs.append(subprocess.Popen(cmd, stdout=open(lg,'w'), stderr=subprocess.STDOUT))    print(f'{nshards}개 프로세스 시작 (GPU 0~{nshards-1}), 목표 {n}개\n')    t0 = time.time()    try:        while any(p.poll() is None for p in procs):            time.sleep(10)            clear_output(wait=True)            done = len(list(CACHE.glob('*/latent.npy')))            print(f'경과 {(time.time()-t0)/60:.1f}분 | 완성된 샘플 {done}개')            for sh, lg in enumerate(logs):                try: last = [l for l in open(lg).read().splitlines() if l.startswith('[shard')][-1]                except Exception: last = '(시작 중)'                alive = '실행중' if procs[sh].poll() is None else '종료'                print(f'  GPU {sh} [{alive}] {last[:110]}')    except KeyboardInterrupt:        for p in procs: p.terminate()        print('중단됨. 다시 실행하면 이어서 진행된다.'); return    clear_output(wait=True)    done = len(list(CACHE.glob('*/latent.npy')))    print(f'전처리 완료: {done}개, {(time.time()-t0)/60:.1f}분')    for sh, lg in enumerate(logs):        try: print(f'  GPU {sh}:', [l for l in open(lg).read().splitlines() if '완료' in l][-1][:120])        except Exception: passrun_preprocess()

In [ ]:
# 통과한 데이터 확인items = sorted([d for d in CACHE.iterdir() if (d/'latent.npy').exists() and (d/'cond.npy').exists()])metas = [json.load(open(d/'meta.json')) for d in items if (d/'meta.json').exists()]print(f'학습 샘플 {len(items)}개')if metas:    ch = np.array([m['chamfer'] for m in metas]); fc = np.array([m['faces'] for m in metas])    print(f'  VAE 재현오차: 평균 {ch.mean():.4f}, 최대 {ch.max():.4f}')    print(f'  면 수: 중앙값 {int(np.median(fc)):,}')    print('  예시:', [m['name'][:20] for m in metas[:5]])steps_per_epoch = max(len(items)//CFG['BATCH'], 1)print(f'\n에폭당 {steps_per_epoch} 스텝 x {CFG["EPOCHS"]} 에폭 = 총 {steps_per_epoch*CFG["EPOCHS"]} 스텝')print(f'예상 소요: {steps_per_epoch*CFG["EPOCHS"]*4.1/3600:.1f}시간 (1장 기준)')

In [ ]:
# 렌더 눈으로 확인 (뷰 규칙이 맞는지, 배경이 잘 빠졌는지)# 주의: 초기 파일럿이 만든 캐시에는 views/ 가 없다. 학습에는 문제 없고 미리보기만 못 한다.from PIL import Imagewithviews = [d for d in items if (d/'views'/'front.png').exists()]print(f'뷰 이미지가 있는 샘플 {len(withviews)}/{len(items)}개')if withviews:    for d in withviews[:2]:        ims = [Image.open(d/'views'/f'{k}.png') for k in ['front','left','back','right']]        c = Image.new('RGBA', (4*260, 260), (60,60,60,255))        for i, im in enumerate(ims):            r = im.resize((256,256)); c.paste(r, (i*260,0), r)        nm = json.load(open(d/'meta.json'))['name'] if (d/'meta.json').exists() else d.name        print(f'{nm}  | front / left / back / right')        display(c)else:    print('뷰 이미지가 있는 샘플이 없다. 전처리 셀을 한 번 더 돌리면 생긴다.')

## 3. 모델 로드추론과 완전히 같은 전처리를 쓰기 위해 파이프라인을 통째로 불러서그 안의 `model`(DiT), `conditioner`(DINO), `scheduler` 를 그대로 쓴다.

In [ ]:
from hy3dgen.shapegen import Hunyuan3DDiTFlowMatchingPipelineDEV = 'cuda:0'pipe = Hunyuan3DDiTFlowMatchingPipeline.from_pretrained(    'tencent/Hunyuan3D-2mv', subfolder='hunyuan3d-dit-v2-mv', variant='fp16', device=DEV)dit, conditioner, sched = pipe.model, pipe.conditioner, pipe.schedulerDT = next(dit.parameters()).dtype          # ★ dit.dtype 은 존재하지 않는다dit = dit.to(DEV).eval(); conditioner = conditioner.to(DEV).eval()for p in dit.parameters(): p.requires_grad_(False)for p in conditioner.parameters(): p.requires_grad_(False)print(f'DiT {sum(p.numel() for p in dit.parameters())/1e9:.2f}B, dtype {DT}')print(f'double blocks {len(dit.double_blocks)} | single blocks {len(dit.single_blocks)} | hidden {dit.hidden_size}')print('scheduler', type(sched).__name__, '| num_train_timesteps', sched.config.num_train_timesteps)

## 4. 데이터셋

In [ ]:
from torch.utils.data import Dataset, DataLoaderclass ShapeDS(Dataset):    def __init__(self, dirs): self.items = dirs    def __len__(self): return len(self.items)    def __getitem__(self, i):        d = self.items[i]        return (torch.from_numpy(np.load(d/'latent.npy')).float(),                torch.from_numpy(np.load(d/'cond.npy')).float())ds = ShapeDS(items)dl = DataLoader(ds, batch_size=CFG['BATCH'], shuffle=True, num_workers=4,                drop_last=True, pin_memory=True, persistent_workers=True)print(f'샘플 {len(ds)}개, 배치 {CFG["BATCH"]}, 에폭당 {len(dl)} 스텝')_l, _c = ds[0]; print('latent', tuple(_l.shape), '| cond', tuple(_c.shape))

## 5. LoRA 주입실제 모듈 이름 (소스에서 확인):- `DoubleStreamBlock` ×16 : `img_attn.qkv`, `img_attn.proj`, `txt_attn.qkv`, `txt_attn.proj`- `SingleStreamBlock` ×32 : `linear1`, `linear2`**주의**: single block 의 두 Linear 는 attention 과 MLP 가 융합돼 있다.`linear1` 출력이 `[qkv | mlp_in]` 이므로, attention 만 노리려면 앞 `3*hidden` 채널만건드려야 한다. 아래 `attn_out` 이 그 처리를 한다.

In [ ]:
import torch.nn as nnfrom torch.utils.checkpoint import checkpoint# ★ LoRA 의 A/B 는 반드시 fp32.#   base 가 fp16 이라고 따라 쓰면 AdamW 의 eps(1e-8) 가 fp16 최소 정규값(6.1e-5)#   아래라 0 으로 언더플로해서 1스텝 만에 NaN 이 된다. 실측으로 확인함.class LoRALinear(nn.Module):    def __init__(self, base, r=8, alpha=16, attn_out=None):        super().__init__(); self.base = base        for p in self.base.parameters(): p.requires_grad_(False)        dv = base.weight.device        self.out_slice = attn_out        out_f = attn_out or base.out_features        self.A = nn.Parameter(torch.zeros(r, base.in_features, device=dv, dtype=torch.float32))        self.B = nn.Parameter(torch.zeros(out_f, r, device=dv, dtype=torch.float32))        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))        self.r, self.alpha = r, alpha        self.scale = alpha / r    def forward(self, x):        # 가중치는 fp32 로 보관하되 연산은 base dtype 으로.        # x.float() 로 올리면 활성값 전체가 fp32 가 되어 약 4배 느려진다.        y = self.base(x)        A = self.A.to(x.dtype); B = self.B.to(x.dtype)        d = (x @ A.T) @ B.T * self.scale        if self.out_slice:            y = y.clone(); y[..., :self.out_slice] = y[..., :self.out_slice] + d            return y        return y + ddef inject_lora(dit, rank, alpha, targets='attn'):    hid, n = dit.hidden_size, 0    for blk in dit.double_blocks:        for at in (blk.img_attn, blk.txt_attn):            for nm in ('qkv','proj'):                setattr(at, nm, LoRALinear(getattr(at, nm), rank, alpha)); n += 1        if targets == 'attn+mlp':            for mlp in (blk.img_mlp, blk.txt_mlp):                for idx in (0, 2): mlp[idx] = LoRALinear(mlp[idx], rank, alpha); n += 1    for blk in dit.single_blocks:        blk.linear1 = LoRALinear(blk.linear1, rank, alpha,                                 attn_out=(3*hid if targets == 'attn' else None)); n += 1        blk.linear2 = LoRALinear(blk.linear2, rank, alpha); n += 1    return nclass CkptWrap(nn.Module):    def __init__(self, m): super().__init__(); self.m = m    def forward(self, *a, **kw):        ks = list(kw); vs = list(a) + [kw[k] for k in ks]; na = len(a)        return checkpoint(lambda *v: self.m(*v[:na], **dict(zip(ks, v[na:]))),                          *vs, use_reentrant=False)if not any(isinstance(m, LoRALinear) for m in dit.modules()):    n = inject_lora(dit, CFG['RANK'], CFG['ALPHA'], CFG['TARGETS'])    dit.double_blocks = nn.ModuleList([CkptWrap(b) for b in dit.double_blocks])    dit.single_blocks = nn.ModuleList([CkptWrap(b) for b in dit.single_blocks])    print(f'LoRA {n}개 Linear 주입 + gradient checkpointing 적용')else:    print('이미 주입되어 있음 (셀 재실행)')lora_params = [p for p in dit.parameters() if p.requires_grad]tot = sum(p.numel() for p in dit.parameters())print(f'학습 파라미터 {sum(p.numel() for p in lora_params)/1e6:.2f}M / 전체 {tot/1e9:.2f}B '      f'({sum(p.numel() for p in lora_params)/tot*100:.2f}%)')

## 6. 학습

In [ ]:
steps_total = CFG['EPOCHS'] * max(len(dl), 1)opt = torch.optim.AdamW(lora_params, lr=CFG['LR'], weight_decay=0.01, betas=(0.9, 0.99))schedlr = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=CFG['LR'],                                              total_steps=max(steps_total,1), pct_start=0.05)start_step, hist = 0, []if CFG['RESUME'] and (CKPT/'lora_last.pt').exists():    ck = torch.load(CKPT/'lora_last.pt', map_location='cpu', weights_only=False)    sd = {k: v for k, v in ck['lora'].items()}    missing = dit.load_state_dict(sd, strict=False)    start_step, hist = ck.get('step', 0), list(ck.get('hist', []))    for _ in range(start_step): schedlr.step()    print(f'이어서 학습: step {start_step} 부터 (저장된 loss {ck.get("loss")})')UNCOND = Nonedef train_step(x0, cond):    global UNCOND    B = x0.shape[0]    # ★ 타임스텝은 fp32. 모델 안에서 1000 을 또 곱하므로 fp16 이면 NaN.    sig32 = torch.rand(B, device=DEV, dtype=torch.float32)    s = sig32.to(x0.dtype).view(B, 1, 1)    noise = torch.randn_like(x0)    x_t = s*noise + (1-s)*x0    target = noise - x0    if CFG['DROP_COND'] > 0:        m = torch.rand(B, device=DEV) < CFG['DROP_COND']        if m.any():            if UNCOND is None or UNCOND.shape[0] < B:                with torch.no_grad():                    UNCOND = conditioner.unconditional_embedding(                        B, view_idxs=[(0,1,2,3)]*B)['main'].to(DEV, x0.dtype)            cond = cond.clone(); cond[m] = UNCOND[:B][m]    pred = dit(x_t, sig32*sched.config.num_train_timesteps, {'main': cond})    return torch.nn.functional.mse_loss(pred.float(), target.float())def save_ckpt(step, loss):    sd = {k: v.detach().cpu() for k, v in dit.state_dict().items() if ('.A' in k or '.B' in k)}    obj = {'step': step, 'loss': loss, 'cfg': CFG, 'lora': sd, 'hist': hist}    torch.save(obj, CKPT/f'lora_{step:06d}.pt'); torch.save(obj, CKPT/'lora_last.pt')print(f'총 {steps_total} 스텝 (남은 {steps_total-start_step}) | 체크포인트 {CKPT}')

In [ ]:
import matplotlib.pyplot as pltfrom IPython.display import clear_outputfrom tqdm.auto import tqdmPLOT_EVERY = 25step, t0 = start_step, time.time()dit.train()pbar = tqdm(total=steps_total, initial=start_step, desc='training', unit='step', dynamic_ncols=True)try:    done = False    for ep in range(CFG['EPOCHS']):        if done: break        for x0, cond in dl:            x0 = x0.to(DEV, DT, non_blocking=True)            cond = cond.to(DEV, DT, non_blocking=True)            loss = train_step(x0, cond)            loss.backward()            gn = torch.nn.utils.clip_grad_norm_(lora_params, 1.0)            opt.step(); schedlr.step(); opt.zero_grad(set_to_none=True)            hist.append(loss.item()); step += 1            pbar.update(1)            pbar.set_postfix({'ep': f'{ep+1}/{CFG["EPOCHS"]}',                              'loss': f'{np.mean(hist[-20:]):.4f}',                              'lr': f'{schedlr.get_last_lr()[0]:.2e}',                              'gnorm': f'{float(gn):.2f}',                              'GiB': f'{torch.cuda.max_memory_allocated()/2**30:.1f}'})            if step % PLOT_EVERY == 0 or step >= steps_total:                clear_output(wait=True); display(pbar.container) if hasattr(pbar,'container') else None                k = min(25, max(len(hist)//5, 2))                plt.figure(figsize=(9,3.2))                plt.plot(hist, lw=0.7, alpha=0.3)                if len(hist) > k:                    plt.plot(np.arange(k-1,len(hist)), np.convolve(hist,np.ones(k)/k,'valid'), lw=2)                el = time.time()-t0; per = el/max(step-start_step,1)                plt.xlabel('step'); plt.ylabel('loss'); plt.grid(alpha=0.3)                plt.title(f'step {step}/{steps_total} ({step/steps_total*100:.0f}%)  '                          f'loss {np.mean(hist[-20:]):.4f}  {per:.2f}s/step  '                          f'경과 {el/60:.0f}분  남음 {per*(steps_total-step)/60:.0f}분')                plt.tight_layout(); plt.show()            if step % CFG['CKPT_EVERY'] == 0: save_ckpt(step, float(np.mean(hist[-20:])))            if step >= steps_total: done = True; breakfinally:    pbar.close()save_ckpt(step, float(np.mean(hist[-20:])))print(f'학습 완료. {step} 스텝, {(time.time()-t0)/60:.1f}분, 최종 loss {np.mean(hist[-20:]):.4f}')

## 7. 평가같은 4뷰 입력을 base 와 LoRA 로 각각 생성해 비교한다.LoRA 를 끄는 것은 `scale=0` 으로 처리한다.**입력 이미지는 반드시 배경을 제거해야 한다.** 흰 배경 RGB 를 그대로 넣으면배경이 물체로 인식돼 정면이 평평한 판때기로 나온다.

In [ ]:
from PIL import Imagefrom bgremove import remove_backgrounddef set_lora_scale(dit, s):    for m in dit.modules():        if isinstance(m, LoRALinear): m.scale = s * (m.alpha / m.r)@torch.no_grad()def generate(img_dict, steps=50, octree=380, seed=1234):    dit.eval()    return pipe(image=img_dict, num_inference_steps=steps, octree_resolution=octree,                num_chunks=20000, generator=torch.manual_seed(seed), output_type='trimesh')[0]TEST = HOME/'inputs/sample1_character'timgs = {}for k in ['front','left','back','right']:    f = TEST/f'{k}.png'    if f.exists(): timgs[k] = remove_background(Image.open(f), 'auto')print('테스트 뷰:', list(timgs))for k, v in timgs.items():    a = np.asarray(v)[..., 3]    print(f'   {k}: 투명 비율 {np.mean(a==0):.2f}   (0.00 이면 배경 제거 실패 → 판때기가 나온다)')

In [ ]:
set_lora_scale(dit, 0.0); base_mesh = generate(timgs); base_mesh.export(DATA/'eval_base.glb')set_lora_scale(dit, 1.0); lora_mesh = generate(timgs); lora_mesh.export(DATA/'eval_lora.glb')print('base faces', len(base_mesh.faces), '| lora faces', len(lora_mesh.faces))# 학습이 실제로 반영됐는지 수치로 확인from scipy.spatial import cKDTreedef npts(m, n=40000):    v, _ = trimesh.sample.sample_surface(m, n)    v = np.asarray(v) - m.bounds.mean(0); return v/np.abs(v).max()import trimeshA, B = npts(base_mesh), npts(lora_mesh)print(f'base vs LoRA chamfer: {(cKDTree(B).query(A)[0].mean()+cKDTree(A).query(B)[0].mean())/2:.5f}')w = torch.cat([p.detach().float().flatten() for n,p in dit.named_parameters()               if p.requires_grad and n.endswith('.B')])print(f'LoRA B 가중치: absmax {w.abs().max():.4f}, std {w.std():.5f}  (0 이면 학습 안 된 것)')

In [ ]:
from matplotlib.collections import PolyCollectiondef rotY(P, deg):    a = np.radians(deg); c, s = np.cos(a), np.sin(a)    return P @ np.array([[c,0,s],[0,1,0],[-s,0,c]]).Tdef show(meshes, titles):    views = [('front',(0,1,2),1), ('back',(0,1,2),-1), ('left side',(2,1,0),-1), ('3/4',None,None)]    light = np.array([0.3,0.5,0.8]); light /= np.linalg.norm(light)    fig, axs = plt.subplots(len(meshes), 4, figsize=(15,4*len(meshes)), facecolor='#333')    axs = np.atleast_2d(axs)    for r, (m, t) in enumerate(zip(meshes, titles)):        mm = m.simplify_quadric_decimation(face_count=200000) if len(m.faces) > 200000 else m        V = np.asarray(mm.vertices); V = V - mm.bounds.mean(0); V = V/np.abs(V).max()        N = mm.face_normals        for c, (vt, ax_, flip) in enumerate(views):            ax = axs[r,c]; P, NN = V, N            if ax_ is None: P, NN = rotY(V,-35), rotY(N,-35); hx,vy,dz,flip = 0,1,2,1            else: hx, vy, dz = ax_            tri = P[mm.faces]; order = np.argsort(tri[:,:,dz].mean(1)*flip)            sh = np.clip(np.abs(NN@light), 0.15, 1)            ax.add_collection(PolyCollection(tri[order][:,:,[hx,vy]],                              facecolors=plt.cm.gray(sh[order]), edgecolors='none', antialiased=False))            ax.set_xlim(-1.05,1.05); ax.set_ylim(-1.05,1.05); ax.set_aspect('equal'); ax.axis('off')            ax.set_title((t+' | ' if c==0 else '')+vt, color='w', fontsize=10, loc='left')    plt.tight_layout(); plt.show()show([base_mesh, lora_mesh], ['base', 'LoRA'])

## 8. 다음 단계- 지금은 Stage 1 에 해당한다. 이 체크포인트에서 이어서 `hiprec_chibi.json` 으로  추가 학습하면 계획서의 Stage 2 (SD/chibi 비율 강화) 가 된다.  `CFG['LIST']` 만 바꾸고 `RESUME=True` 로 두면 된다.- **VRoid 는 쓸 수 없다.** pixiv VRoid Studio 약관 7조·13조가  "아바타·3D 모델을 생성·출력하는 애플리케이션 제작" 을 명시적으로 금지한다.  VRoid Hub 크롤링도 약관 3조 위반. CharacterGen/StdGEN 의 Anime3D 도 같은 이유로 미공개.- Objaverse 는 CC-BY 라 **출처 표기 의무**가 있고, 원작 IP 팬메이드가 섞여 있어  상업 배포 시 별도 검토가 필요하다.- 아직 미확정: 뷰 규칙. 비대칭 물체로 재보면 `left=90도` 가 19% 정확했지만  한 번만 측정했다. 시드를 바꿔 여러 번 재면 확정된다.